# Module 7 · Prompt Golf — Class Competition

Write the **shortest prompt** that produces the correct answer. Fewer tokens = higher score.

---

## Rules

| | |
|---|---|
| Scoring | `1000 / prompt_token_count` if correct, `0` if wrong |
| Retries | Unlimited — only your **best score per challenge** counts |
| Challenges | 5 total — do them in any order |
| Win | Highest **total score** across all 5 challenges |

**Example:** A correct answer with a 10-token prompt scores `1000/10 = 100`. A correct answer with a 5-token prompt scores `200`. Wrong answer = `0`.

---
> **Setup:** Run cells 1 and 2 first, then set your email and the instructor URL in cell 3.

In [1]:
import os, json, re, time, getpass, requests
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemini-3.1-flash-lite'
DEFAULT_MAX = 8192

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.0)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            elif '500' in msg or 'INTERNAL' in msg:
                wait = 10 * (attempt + 1)
                print(f'  Server error (500) — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen   = client.models.generate_content
_raw_count = client.models.count_tokens
client.models.generate_content = lambda *a, **kw: _call_with_retry(_raw_gen,   *a, **kw)
client.models.count_tokens     = lambda *a, **kw: _call_with_retry(_raw_count, *a, **kw)

print('Setup complete. Model:', MODEL)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Setup complete. Model: gemini-3.1-flash-lite


In [2]:
# ✏️  Fill in your details and the URL your instructor shared
MY_EMAIL       = "24d148@psgitech.ac.in"  # <-- change this
INSTRUCTOR_URL = "https://nondiligently-unfederated-fredric.ngrok-free.dev"    # <-- paste the URL from your instructor

# Quick connection check
try:
    r = requests.get(f"{INSTRUCTOR_URL}/challenges", timeout=5, headers={"ngrok-skip-browser-warning": "true"})
    r.raise_for_status()
    print(f'Connected to leaderboard server. {len(r.json())} challenges available.')
except Exception as e:
    print(f'Could not reach server: {e}')
    print('Check INSTRUCTOR_URL and make sure the server is running.')

Connected to leaderboard server. 5 challenges available.


In [18]:
# View all challenges
resp = requests.get(f"{INSTRUCTOR_URL}/challenges", timeout=5, headers={"ngrok-skip-browser-warning": "true"})
challenges = {c['id']: c for c in resp.json()}

print('=' * 70)
print('  PROMPT GOLF — ALL CHALLENGES')
print('  YOUR PROMPT must provide all instructions.')
print('  The task below is RAW INPUT only — your prompt does all the work.')
print('=' * 70)
for c in resp.json():
    print(f"\nQ{c['number']} [{c['id']}]  {c['title']}")
    print(f"Concept     : {c['concept']}")
    print(f"Description : {c['description']}")
    print(f"\nInput       :\n{c['task']}")
    print('-' * 70)


  PROMPT GOLF — ALL CHALLENGES
  YOUR PROMPT must provide all instructions.
  The task below is RAW INPUT only — your prompt does all the work.

Q1 [q1]  The Ambiguous Sentiment
Concept     : Few-shot prompting — label calibration with ambiguous inputs
Description : A text classifier needs to pick EXACTLY ONE label from: POSITIVE, NEGATIVE, NEUTRAL, MIXED. The tricky part: the input below contains BOTH a genuine compliment AND a real complaint — making it MIXED, not POSITIVE or NEGATIVE. Zero-shot prompts almost always mis-classify it as NEGATIVE because the complaint is more salient. You need to teach the model the exact definition of MIXED with examples before it sees the input. Your output must be the single word MIXED and nothing else.

Input       :
Review: "The onboarding team was incredibly patient and helpful — best support experience I've ever had. But the software itself crashes every time I try to export a PDF, which completely kills my workflow."
---------------------------

In [36]:
# ✏️  Choose a challenge and write your prompt
CHALLENGE_ID = "q3"   # q1 / q2 / q3 / q4 / q5

MY_PROMPT = """5 lines, LABEL: value, no markdown, nothing else. ROOT_CAUSE = short phrase not a sentence. SEVERITY = inferred from impact, not stated in text.."""
# ── Run ────────────────────────────────────────────
challenge = challenges[CHALLENGE_ID]
full_prompt = MY_PROMPT + "\n\n" + challenge['task']

prompt_tokens = client.models.count_tokens(model=MODEL, contents=MY_PROMPT).total_tokens

response = client.models.generate_content(
    model=MODEL,
    contents=full_prompt,
    config=cfg(temperature=0.0)
)
output = get_text(response)

# ── Judge ──────────────────────────────────────────────
judge_prompt = (
    challenge['judge_instruction'] + "\n\n"
    "Student output: " + output + "\n"
    "Reply only YES or NO."
)
verdict = get_text(client.models.generate_content(
    model=MODEL,
    contents=judge_prompt,
    config=cfg(temperature=0.0, max_output_tokens=8192)
))
correct = "YES" in verdict.upper()
score = round(1000 / prompt_tokens, 2) if correct else 0

# ── Print result ──────────────────────────────────────────
print("=" * 55)
print(f"Challenge : Q{challenge['number']} — {challenge['title']}")
print(f"Concept   : {challenge['concept']}")
print(f"Output    : {output[:200]}")
print(f"Correct   : {'✅ YES' if correct else '❌ NO'}")
print(f"Tokens    : {prompt_tokens}")
print(f"Score     : {score}")
print("=" * 55)

# ── Submit ──────────────────────────────────────────────
try:
    sub = requests.post(
        f"{INSTRUCTOR_URL}/submit",
        headers={"ngrok-skip-browser-warning": "true"},
        json={
            "email": MY_EMAIL,
            "challenge_id": CHALLENGE_ID,
            "score": score,
            "prompt_tokens": prompt_tokens,
            "prompt": MY_PROMPT,
            "output": output,
        },
        timeout=5
    ).json()
    if sub.get("improved"):
        print(f"\n🏆 New best for {CHALLENGE_ID}! Rank: #{sub['rank']} of {sub['total_students']}")
    else:
        print(f"\nNo improvement (best stays {sub['best_score']}). Rank: #{sub['rank']} of {sub['total_students']}")
except Exception as e:
    print(f"Could not submit to leaderboard: {e}")


Challenge : Q3 — The Strict Extractor
Concept   : Output format control — multi-field structured extraction with edge-case fields
Output    : INCIDENT_ID: INC-2047
ROOT_CAUSE: Misconfigured database connection pool
SEVERITY: High
IMPACT: 4,200 customers
REVENUE_LOSS: $38,000
Correct   : ❌ NO
Tokens    : 37
Score     : 0

No improvement (best stays 0.0). Rank: #11 of 24


In [ ]:
# View current leaderboard
rows = requests.get(f"{INSTRUCTOR_URL}/leaderboard_json", timeout=5, headers={"ngrok-skip-browser-warning": "true"}).json()

challenge_ids = [c['id'] for c in challenges.values()]

# Header
print(f"{'#':<4} {'Student':<30} {'Total':>7}  " + "  ".join(f"{cid.upper():>6}" for cid in challenge_ids))
print('-' * (4 + 30 + 9 + 10 * len(challenge_ids)))

medals = ['1st', '2nd', '3rd']
for i, row in enumerate(rows):
    rank = medals[i] if i < 3 else str(i + 1)
    scores_str = "  ".join(
        f"{row['challenges'].get(cid, {}).get('score', '—'):>6}"
        for cid in challenge_ids
    )
    print(f"{rank:<4} {row['email']:<30} {row['total']:>7}  {scores_str}")